In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [4]:
train_df = pd.read_csv("../data/raw/train.csv")
train_df.shape

/var/folders/gw/6hw2mnqn2p3c3ylx1sjb3gsw0000gn/T/ipykernel_51720/884536798.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("../data/raw/train.csv")


(100000, 28)

In [5]:
def convert_credit_history_age(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value)
    
    years = 0
    months = 0
    
    if "Years" in value:
        years = int(value.split(" Years")[0])
    
    if "Months" in value:
        months_part = value.split("and ")[-1].split(" Months")[0]
        months = int(months_part)
    
    return years * 12 + months


def clean_data(df):
    df = df.copy()
    
    # placeholder 처리
    placeholders = ["_", "_______", "!@9#%8"]
    df = df.replace(placeholders, np.nan)
    
    # 숫자인데 object인 컬럼 변환
    numeric_like_cols = [
        "Age",
        "Annual_Income",
        "Num_of_Loan",
        "Num_of_Delayed_Payment",
        "Changed_Credit_Limit",
        "Outstanding_Debt",
        "Amount_invested_monthly",
        "Monthly_Balance",
    ]
    
    for col in numeric_like_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace("_", "", regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Credit_History_Age 변환
    if "Credit_History_Age" in df.columns:
        df["Credit_History_Age_Months"] = df["Credit_History_Age"].apply(
            convert_credit_history_age
        )
        df = df.drop(columns=["Credit_History_Age"])
    
    # ID성 컬럼 제거
    drop_cols = ["ID", "Name", "SSN"]
    df = df.drop(columns=[col for col in drop_cols if col in df.columns])
    
    # 이상치 처리: 말이 안 되는 값은 NaN으로
    if "Age" in df.columns:
        df.loc[~df["Age"].between(18, 100), "Age"] = np.nan
    
    if "Num_Bank_Accounts" in df.columns:
        df.loc[~df["Num_Bank_Accounts"].between(0, 20), "Num_Bank_Accounts"] = np.nan
    
    if "Num_Credit_Card" in df.columns:
        df.loc[~df["Num_Credit_Card"].between(0, 20), "Num_Credit_Card"] = np.nan
    
    if "Interest_Rate" in df.columns:
        df.loc[~df["Interest_Rate"].between(0, 100), "Interest_Rate"] = np.nan
    
    if "Num_of_Loan" in df.columns:
        df.loc[~df["Num_of_Loan"].between(0, 20), "Num_of_Loan"] = np.nan
    
    if "Delay_from_due_date" in df.columns:
        df.loc[df["Delay_from_due_date"] < 0, "Delay_from_due_date"] = np.nan
    
    if "Num_of_Delayed_Payment" in df.columns:
        df.loc[~df["Num_of_Delayed_Payment"].between(0, 100), "Num_of_Delayed_Payment"] = np.nan
    
    if "Num_Credit_Inquiries" in df.columns:
        df.loc[~df["Num_Credit_Inquiries"].between(0, 100), "Num_Credit_Inquiries"] = np.nan
    
    if "Monthly_Balance" in df.columns:
        df.loc[df["Monthly_Balance"] < 0, "Monthly_Balance"] = np.nan
    
    return df

In [6]:
clean_df = clean_data(train_df)

print("before:", train_df.shape)
print("after:", clean_df.shape)

clean_df.head()

before: (100000, 28)
after: (100000, 25)


,Customer_ID,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Credit_History_Age_Months
0,CUS_0xd40,January,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3.0,7.0,11.27,4.0,NaN,809.98,26.822620,No,49.574949,80.415295,High_spent_Small_value_payments,312.494089,Good,265.0
1,CUS_0xd40,February,23.0,Scientist,19114.12,NaN,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",NaN,NaN,11.27,4.0,Good,809.98,31.944960,No,49.574949,118.280222,Low_spent_Large_value_payments,284.629162,Good,NaN
2,CUS_0xd40,March,NaN,Scientist,19114.12,NaN,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3.0,7.0,NaN,4.0,Good,809.98,28.609352,No,49.574949,81.699521,Low_spent_Medium_value_payments,331.209863,Good,267.0
3,CUS_0xd40,April,23.0,Scientist,19114.12,NaN,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5.0,4.0,6.27,4.0,Good,809.98,31.377862,No,49.574949,199.458074,Low_spent_Small_value_payments,223.451310,Good,268.0
4,CUS_0xd40,May,23.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6.0,NaN,11.27,4.0,Good,809.98,24.797347,No,49.574949,41.420153,High_spent_Medium_value_payments,341.489231,Good,269.0


In [7]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Customer_ID                100000 non-null  object 
 1   Month                      100000 non-null  object 
 2   Age                        91518 non-null   float64
 3   Occupation                 92938 non-null   object 
 4   Annual_Income              100000 non-null  float64
 5   Monthly_Inhand_Salary      84998 non-null   float64
 6   Num_Bank_Accounts          98665 non-null   float64
 7   Num_Credit_Card            97737 non-null   float64
 8   Interest_Rate              97988 non-null   float64
 9   Num_of_Loan                95655 non-null   float64
 10  Type_of_Loan               88592 non-null   object 
 11  Delay_from_due_date        99409 non-null   float64
 12  Num_of_Delayed_Payment     91630 non-null   float64
 13  Changed_Credit_Limit       979

In [8]:
missing_after = pd.DataFrame({
    "missing_count": clean_df.isnull().sum(),
    "missing_ratio": clean_df.isnull().mean() * 100
})

missing_after[missing_after["missing_count"] > 0].sort_values(
    by="missing_count",
    ascending=False
)

,missing_count,missing_ratio
Credit_Mix,20195,20.195
Monthly_Inhand_Salary,15002,15.002
Type_of_Loan,11408,11.408
Credit_History_Age_Months,9030,9.030
Age,8482,8.482
Num_of_Delayed_Payment,8370,8.370
Payment_Behaviour,7600,7.600
Occupation,7062,7.062
Amount_invested_monthly,4479,4.479
Num_of_Loan,4345,4.345


In [9]:
target = "Credit_Score"

X = clean_df.drop(columns=[target])
y = clean_df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("numeric:", numeric_features)
print("categorical:", categorical_features)

numeric: ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_Months']
categorical: ['Customer_ID', 'Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
